# GenRec Phase-2 — Kaggle 2x T4

Settings BEFORE Run All: **Accelerator = GPU T4 x2**, **Internet = ON**.

Upload alongside this notebook: `data_prep.py`, `verbalize.py`, `model.py`, `train_phase2.py`, `eval.py`, `requirements.txt`.

Flow: installs -> GPU assert -> data -> smoke run -> full run -> test eval.
If the session dies, re-run from the full-run cell with `--resume` (checkpoints every 500 steps).

In [ ]:
!pip install -q peft bitsandbytes accelerate

In [ ]:
import torch
assert torch.cuda.is_available(), "no GPU"
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("GPU", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() == 2, "expected 2 GPUs"
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)

In [ ]:
# data: download + build catalog/events/train_cuts (or upload your data/ instead)
!mkdir -p data && (wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O data/ml-1m.zip || wget -q --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-1m.zip -O data/ml-1m.zip)
!python data_prep.py

In [ ]:
# smoke: ~10 min incl. model download. If this OOMs or loss doesn't move, fix before the real run.
!python train_phase2.py --max-examples 400 --epochs 1 --batch 2 --accum 4 --val-users 200 --out runs/smoke

In [ ]:
# full base run (~1.5-2h). Resume after a kill: just re-run this cell (--resume is on).
!python train_phase2.py --resume --out runs/phase2_base

In [ ]:
# test eval (full catalog, leave-one-out) — the M2 gate: beat SASRec MRR 0.3045
import json
import eval as eval_lib
from model import GenRecModel, genrec_scorer, load_tokenizer

tok = load_tokenizer()
catalog_map = {c["item_id"]: c for c in map(json.loads, open("data/catalog.jsonl"))}
_, users = eval_lib.load_all("data")
model, step = GenRecModel.from_ckpt("runs/phase2_base")
m = eval_lib.evaluate(genrec_scorer(model, tok, catalog_map), users, "test")
print(json.dumps({"scorer": "genrec", "ckpt_step": step, **m}, indent=2))